# mzIdentML Results Explorer

This notebook reads a xiFDR/xiSEARCH `.mzid` file and presents its identifications as compact tables for initial inspection.

## Load `.mzid` File

Set `MZID_PATH` to a local mzIdentML results file. The default points to the supplied successful PXD042173 example. Loading is read-only and validates that spectrum-identification records are present.

In [4]:
from pathlib import Path
import warnings
import xml.etree.ElementTree as ET

import pandas as pd

MZID_PATH = Path("pxd_data/PXD042173/results.mzid")
NAMESPACE = "{http://psidev.info/psi/pi/mzIdentML/1.3}"

if not MZID_PATH.is_file():
    raise FileNotFoundError(f"mzIdentML file not found: {MZID_PATH.resolve()}")


def local_name(element):
    return element.tag.removeprefix(NAMESPACE)


records = {
    "proteins": {},
    "peptides": {},
    "evidence": {},
    "spectrum_results": [],
}
parse_complete = True

try:
    for _, element in ET.iterparse(MZID_PATH, events=("end",)):
        name = local_name(element)

        if name == "DBSequence":
            description = next(
                (
                    param.get("value", "")
                    for param in element.findall(f"{NAMESPACE}cvParam")
                    if param.get("name") == "protein description"
                ),
                "",
            )
            records["proteins"][element.get("id")] = {
                "protein_id": element.get("id"),
                "accession": element.get("accession"),
                "name": element.get("name"),
                "length": element.get("length"),
                "description": description,
                "is_decoy": element.get("id", "").startswith("dbseq_decoy"),
            }
            element.clear()

        elif name == "Peptide":
            sequence = element.findtext(f"{NAMESPACE}PeptideSequence", default="")
            modifications = []
            for modification in element.findall(f"{NAMESPACE}Modification"):
                label = next(
                    (
                        param.get("name") or param.get("accession")
                        for param in modification.findall(f"{NAMESPACE}cvParam")
                    ),
                    modification.get("monoisotopicMassDelta", "modification"),
                )
                modifications.append(f"{label}@{modification.get('location', '?')}")
            records["peptides"][element.get("id")] = {
                "peptide_id": element.get("id"),
                "sequence": sequence,
                "modifications": "; ".join(modifications),
            }
            element.clear()

        elif name == "PeptideEvidence":
            records["evidence"][element.get("id")] = dict(element.attrib)
            element.clear()

        elif name == "SpectrumIdentificationResult":
            items = []
            for item in element.findall(f"{NAMESPACE}SpectrumIdentificationItem"):
                scores = {
                    param.get("name", param.get("accession", "score")): param.get("value", "")
                    for param in item.findall(f"{NAMESPACE}cvParam")
                }
                items.append(
                    {
                        **dict(item.attrib),
                        "evidence_refs": [
                            ref.get("peptideEvidence_ref")
                            for ref in item.findall(f"{NAMESPACE}PeptideEvidenceRef")
                        ],
                        "scores": scores,
                    }
                )
            records["spectrum_results"].append(
                {
                    "spectrum_id": element.get("spectrumID", ""),
                    "source_file": Path(element.get("spectraData_ref", "")).name,
                    "items": items,
                }
            )
            element.clear()
except ET.ParseError as error:
    parse_complete = False
    warnings.warn(
        "The mzIdentML file ends before its closing tag. The tables contain only "
        f"fully parsed spectrum results. Parser detail: {error}",
        RuntimeWarning,
        stacklevel=1,
    )

if not records["spectrum_results"]:
    raise ValueError(f"No spectrum-identification records found in {MZID_PATH}")

file_state = "complete" if parse_complete else "partial"
print(
    f"Loaded {MZID_PATH.name} ({file_state}): "
    f"{len(records['spectrum_results']):,} spectra, "
    f"{sum(len(result['items']) for result in records['spectrum_results']):,} PSM items"
)

Loaded results.mzid (partial): 469 spectra, 972 PSM items


/tmp/ipykernel_146720/3601826877.py:99: RuntimeWarning: The mzIdentML file ends before its closing tag. The tables contain only fully parsed spectrum results. Parser detail: unclosed token: line 1193207, column 8
  warnings.warn(


## Transform Into Tables

`crosslink_spectrum_matches` has one row per reported crosslink spectrum match. Alpha and beta peptide information is combined on that row so a single spectrum can be scanned quickly.

`protein_evidence` has one row for each peptide-side to protein mapping. It is useful when a peptide maps to multiple proteins or positions.

`passThreshold` is the mzIdentML threshold result. The displayed tables use only rows where `passThreshold` is `True`; the unfiltered data remains available in `crosslink_spectrum_matches` and `protein_evidence` for later inspection.

Key columns: `scan` is the spectrum scan number; `rank`, `charge`, measured/calculated m/z, and `mass_error_ppm` describe the identification; `xi_score` is xiSEARCH's score. The `*_proteins` and `*_positions` fields in the match table are semicolon-separated summaries; use `protein_evidence` for the individual mappings.

In [7]:
import re
from collections import defaultdict


def as_number(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def peptide_side(peptide_id):
    if peptide_id.endswith("_p0"):
        return "alpha"
    if peptide_id.endswith("_p1"):
        return "beta"
    return "single"


def joined_unique(values):
    return "; ".join(dict.fromkeys(str(value) for value in values if value))


match_rows = []
evidence_rows = []

for result in records["spectrum_results"]:
    matches = defaultdict(list)
    for item in result["items"]:
        crosslink_id = item["scores"].get(
            "cross-link spectrum identification item",
            f"{result['spectrum_id']}:{item.get('rank', '')}",
        )
        matches[crosslink_id].append(item)

    scan_match = re.search(r"scan=(\d+)", result["spectrum_id"])
    scan = int(scan_match.group(1)) if scan_match else None

    for crosslink_id, items in matches.items():
        by_side = defaultdict(list)
        for item in items:
            by_side[peptide_side(item.get("peptide_ref", ""))].append(item)

        side_details = {}
        for side in ("alpha", "beta"):
            side_items = by_side[side]
            peptide_ids = [item.get("peptide_ref") for item in side_items]
            peptides = [records["peptides"].get(peptide_id, {}) for peptide_id in peptide_ids]
            evidence_details = [
                (
                    item,
                    peptide,
                    mapping,
                    records["proteins"].get(mapping.get("dBSequence_ref"), {}),
                )
                for item, peptide in zip(side_items, peptides)
                for evidence_ref in item["evidence_refs"]
                for mapping in [records["evidence"].get(evidence_ref, {})]
            ]

            side_details[side] = {
                "sequence": joined_unique(peptide.get("sequence") for peptide in peptides),
                "modifications": joined_unique(
                    peptide.get("modifications") for peptide in peptides
                ),
                "proteins": joined_unique(
                    protein.get("accession")
                    for _, _, _, protein in evidence_details
                ),
                "positions": joined_unique(
                    f"{protein.get('accession', '')}:{mapping.get('start', '')}-{mapping.get('end', '')}"
                    for _, _, mapping, protein in evidence_details
                ),
            }

            for item, peptide, mapping, protein in evidence_details:
                evidence_rows.append(
                    {
                        "crosslink_spectrum_id": crosslink_id,
                        "spectrum_id": result["spectrum_id"],
                        "peptide_side": side,
                        "passThreshold": item.get("passThreshold", "false").lower() == "true",
                        "peptide_sequence": peptide.get("sequence"),
                        "modifications": peptide.get("modifications"),
                        "protein_accession": protein.get("accession"),
                        "protein_name": protein.get("name"),
                        "protein_description": protein.get("description"),
                        "protein_start": as_number(mapping.get("start")),
                        "protein_end": as_number(mapping.get("end")),
                        "is_decoy": mapping.get("isDecoy", "false").lower() == "true",
                    }
                )

        primary_item = items[0]
        experimental_mz = as_number(primary_item.get("experimentalMassToCharge"))
        calculated_mz = as_number(primary_item.get("calculatedMassToCharge"))
        mass_error_ppm = (
            (experimental_mz - calculated_mz) / calculated_mz * 1_000_000
            if experimental_mz is not None and calculated_mz not in (None, 0)
            else None
        )
        match_rows.append(
            {
                "source_file": result["source_file"],
                "scan": scan,
                "crosslink_spectrum_id": crosslink_id,
                "rank": as_number(primary_item.get("rank")),
                "passThreshold": primary_item.get("passThreshold", "false").lower() == "true",
                "charge": as_number(primary_item.get("chargeState")),
                "experimental_mz": experimental_mz,
                "calculated_mz": calculated_mz,
                "mass_error_ppm": mass_error_ppm,
                "xi_score": as_number(primary_item["scores"].get("xi:score")),
                "peptide_alpha": side_details["alpha"]["sequence"],
                "peptide_beta": side_details["beta"]["sequence"],
                "alpha_modifications": side_details["alpha"]["modifications"],
                "beta_modifications": side_details["beta"]["modifications"],
                "alpha_proteins": side_details["alpha"]["proteins"],
                "beta_proteins": side_details["beta"]["proteins"],
                "alpha_positions": side_details["alpha"]["positions"],
                "beta_positions": side_details["beta"]["positions"],
            }
        )

crosslink_spectrum_matches = pd.DataFrame(match_rows).sort_values(
    ["passThreshold", "xi_score"], ascending=[False, False], na_position="last"
)
protein_evidence = pd.DataFrame(evidence_rows).sort_values(
    ["passThreshold", "crosslink_spectrum_id", "peptide_side", "protein_accession"],
    ascending=[False, True, True, True],
    na_position="last",
)

passed_crosslink_spectrum_matches = crosslink_spectrum_matches.loc[
    crosslink_spectrum_matches["passThreshold"]
].copy()
passed_protein_evidence = protein_evidence.loc[
    protein_evidence["passThreshold"]
].copy()

print(
    f"Created {len(crosslink_spectrum_matches):,} crosslink-spectrum match rows and "
    f"{len(protein_evidence):,} peptide-to-protein evidence rows.\n"
    f"Displaying {len(passed_crosslink_spectrum_matches):,} passed matches and "
    f"{len(passed_protein_evidence):,} passed evidence rows."
)

Created 492 crosslink-spectrum match rows and 1,033 peptide-to-protein evidence rows.
Displaying 15 passed matches and 22 passed evidence rows.


## Display Interactive Tables

Both tables below contain only `passThreshold = True` results. The threshold column remains visible as a final confirmation. Use the table controls to search, sort, and page through the data.

In [8]:
from IPython.display import HTML, display
from itables import show


def show_table(title, table, page_length=25):
    display(HTML(f"<h3>{title} ({len(table):,} rows)</h3>"))
    show(
        table,
        paging=True,
        pageLength=page_length,
        scrollX=True,
        scrollY="500px",
        classes="display compact",
        column_filters="header",
    )


show_table("Passed crosslink-spectrum matches", passed_crosslink_spectrum_matches)
show_table("Passed peptide-to-protein evidence", passed_protein_evidence, page_length=50)